# Lab 07: Object Detection Metrics & Semantic Segmentation with Mini U-Net

Welcome to Laboratory 07! In this lab, we advance into dense spatial prediction tasks:
1. **Bounding Box Overlap Metrics**: Implement **Intersection over Union (IoU)** from scratch for object detection bounding boxes.
2. **Mini U-Net Architecture**: Build a deep encoder-decoder segmentation architecture featuring **skip connections** and transposed convolutions (`ConvTranspose2d`).
3. **Dense Pixel-Wise Mask Prediction**: Train and evaluate semantic segmentation on synthetic binary spatial targets.


## 1. Technical Preliminaries & Imports


In [ ]:
# Import PyTorch and visualization libraries
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

# Reproducibility seeds
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Active Device:', device)


## 2. Object Detection Evaluation: Intersection over Union (IoU)

### Mathematical Formulation
Given two bounding boxes $A = [x_1^A, y_1^A, x_2^A, y_2^A]$ and $B = [x_1^B, y_1^B, x_2^B, y_2^B]$:
* **Intersection Coordinates**:
  $$x_{\text{left}} = \max(x_1^A, x_1^B), \quad y_{\text{top}} = \max(y_1^A, y_1^B)$$
  $$x_{\text{right}} = \min(x_2^A, x_2^B), \quad y_{\text{bottom}} = \min(y_2^A, y_2^B)$$
* **Intersection Area**: $\text{Area}_{\cap} = \max(0, x_{\text{right}} - x_{\text{left}}) \times \max(0, y_{\text{bottom}} - y_{\text{top}})$
* **Union Area**: $\text{Area}_{\cup} = \text{Area}(A) + \text{Area}(B) - \text{Area}_{\cap}$
* **IoU Score**: $\text{IoU} = \frac{\text{Area}_{\cap}}{\text{Area}_{\cup}} \in [0, 1]$.


### Helper Function: `compute_iou`
The function below computes the exact geometric IoU between two bounding boxes.


In [ ]:
def compute_iou(box1: list, box2: list) -> float:
    """Calculates the Intersection over Union (IoU) overlap between two bounding boxes.
    
    Args:
        box1: Bounding box [x1, y1, x2, y2] coordinates
        box2: Bounding box [x1, y1, x2, y2] coordinates
    Returns:
        IoU score as a floating-point value between 0.0 and 1.0.
    """
    # Step 1: Find the top-left and bottom-right coordinates of the intersection rectangle
    x_left = max(box1[0], box2[0])
    y_top = max(box1[1], box2[1])
    x_right = min(box1[2], box2[2])
    y_bottom = min(box1[3], box2[3])
    
    # Step 2: Compute intersection area (ensure non-negative width and height)
    inter_w = max(0, x_right - x_left)
    inter_h = max(0, y_bottom - y_top)
    intersection_area = inter_w * inter_h
    
    # Step 3: Compute individual box areas
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    
    # Step 4: Compute union area and final IoU ratio
    union_area = area1 + area2 - intersection_area
    iou = intersection_area / union_area if union_area > 0 else 0.0
    return iou

# Test IoU computation with sample bounding boxes
box_a = [10, 10, 50, 50]  # Ground truth box (Area = 40x40 = 1600)
box_b = [20, 20, 60, 60]  # Predicted box (Area = 40x40 = 1600, Intersection = 30x30 = 900)

iou_val = compute_iou(box_a, box_b)
print(f'Box A: {box_a} | Box B: {box_b}')
print(f'Computed IoU: {iou_val:.4f} (Expected: 900 / 2300 = {900/2300:.4f})')


## 3. Mini U-Net Architecture for Semantic Segmentation

### Architecture Overview: `MiniUNet`
The U-Net architecture utilizes an **Encoder-Decoder** structure with horizontal **Skip Connections**:
* **Encoder (Contracting Path)**: Extracts high-level semantic features while reducing spatial dimensions via `MaxPool2d(2, 2)`: `(1, 64, 64) -> (16, 64, 64) -> (16, 32, 32)`.
* **Bottleneck**: Deepest representation `Conv2d(16, 32, 3, padding=1)`.
* **Upsampling (Expanding Path)**: `ConvTranspose2d(32, 16, 2, stride=2)` doubles spatial dimensions `(32, 32) -> (64, 64)`.
* **Skip Concatenation**: Concatenates high-resolution encoder features (`16` channels) with upsampled features (`16` channels) along channel dimension (`32` total channels) to preserve fine spatial boundaries.
* **Decoder Head**: `Conv2d(32, 1, 3, padding=1)` followed by `Sigmoid()` to produce pixel probability masks.


In [ ]:
# Define the Mini U-Net Semantic Segmentation Architecture
class MiniUNet(nn.Module):
    """Compact U-Net architecture with Encoder, Bottleneck, Transposed Conv Decoder, and Skip Connections."""
    def __init__(self, in_channels: int = 1, out_channels: int = 1):
        super(MiniUNet, self).__init__()
        
        # 1. Contracting Encoder Path
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2) # 2x spatial downsampling
        
        # 2. Latent Bottleneck Path
        self.bottleneck = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
        
        # 3. Transposed Convolution Upsampling (Expansive Path)
        self.upconv = nn.ConvTranspose2d(in_channels=32, out_channels=16, kernel_size=2, stride=2)
        
        # 4. Decoder Classification Head (Processes concatenated 16 + 16 = 32 channels)
        self.decoder = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=16, out_channels=out_channels, kernel_size=3, padding=1),
            nn.Sigmoid() # Output per-pixel probabilities in [0.0, 1.0]
        )
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Encoder forward pass and capture high-resolution skip features
        skip_features = self.encoder(x)          # Shape: (B, 16, H, W)
        
        # Downsample and process bottleneck
        pooled = self.pool(skip_features)        # Shape: (B, 16, H/2, W/2)
        bottleneck_out = self.bottleneck(pooled) # Shape: (B, 32, H/2, W/2)
        
        # Upsample spatial dimensions
        upsampled = self.upconv(bottleneck_out)  # Shape: (B, 16, H, W)
        
        # Concatenate along channel dimension (dim=1)
        concatenated = torch.cat([upsampled, skip_features], dim=1) # Shape: (B, 32, H, W)
        
        # Final segmentation mask prediction
        mask_pred = self.decoder(concatenated)   # Shape: (B, 1, H, W)
        return mask_pred

# Instantiate MiniUNet model and verify tensor transformations
unet_model = MiniUNet(in_channels=1, out_channels=1).to(device)
sample_input = torch.randn(2, 1, 64, 64).to(device) # Batch of 2 grayscale images of size 64x64
mask_output = unet_model(sample_input)

print('Input Image Tensor Shape:      ', sample_input.shape)
print('Predicted Mask Tensor Shape:   ', mask_output.shape)
assert mask_output.shape == (2, 1, 64, 64), 'U-Net segmentation mask shape mismatch!'
print('[Verification Passed] Mini U-Net successfully outputted dense pixel segmentation masks!')


## 4. Summary & Takeaways
1. **Intersection over Union (IoU)**: Quantifies localization accuracy for bounding boxes in object detection.
2. **U-Net Skip Connections**: Transfer fine-grained spatial information from the encoder directly to the decoder, solving the spatial resolution loss caused by pooling.
3. **Transposed Convolutions**: Learnable upsampling layers that expand feature maps back to the original image dimensions.
